# B2-022-probabilistic-latent-models — Practice p11 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** gaussian-reparameterization

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch

SEED = 20260927
mu = torch.tensor([[0.5, -1.0, 0.0], [2.0, 0.0, -0.5]], requires_grad=True)
logvar = torch.tensor([[0.0, math.log(0.25), 1.0], [-2.0, 0.0, 0.5]], requires_grad=True)
eps = torch.tensor([[1.2, -0.4, 0.0], [-1.0, 0.5, 2.0]])
w = torch.tensor([[1.0, 2.0, 3.0], [-1.0, 0.5, 2.0]])
mu_row = torch.tensor([0.5, -1.0, 0.0])
logvar_row = torch.tensor([0.0, math.log(0.25), 1.0])
ATOL = 1e-6
RTOL = 1e-6

## Solution

`reparameterize` is a pure differentiable expression: $z=\mu+e^{\ell/2}\odot\varepsilon$, so $\partial z/\partial\mu=1$ and $\partial z/\partial\ell=\tfrac12e^{\ell/2}\varepsilon$.
With $L=\sum z$ these are exactly the gradients; with $L=\sum w\odot z$ they are multiplied elementwise by $w$.

In [ ]:
def reparameterize(mu, logvar, eps):
    if not (mu.shape == logvar.shape == eps.shape):
        raise ValueError("mu, logvar, and eps must have identical shapes")
    return mu + torch.exp(0.5 * logvar) * eps


# Probes 1-2
z = reparameterize(mu, logvar, eps)
z.sum().backward()
expected_z = torch.tensor([[1.7, -1.2, 0.0], [1.6321206, 0.5, 2.0680509]])
expected_logvar_grad = torch.tensor([[0.6, -0.1, 0.0], [-0.1839397, 0.25, 1.2840254]])

# Probe 3: fresh leaf copies
mu2 = mu.detach().clone().requires_grad_(True)
logvar2 = logvar.detach().clone().requires_grad_(True)
(w * reparameterize(mu2, logvar2, eps)).sum().backward()

# Probe 4: distribution check
g = torch.Generator().manual_seed(SEED)
big_eps = torch.randn(100_000, 3, generator=g)
mu_big = mu_row.expand(100_000, 3)
logvar_big = logvar_row.expand(100_000, 3)

def distribution_ok(samples):
    mean_ok = bool(((samples.mean(dim=0) - mu_row).abs() <= 0.02).all())
    var_ok = bool(((samples.var(dim=0, correction=0) - logvar_row.exp()).abs() <= 0.05).all())
    return mean_ok and var_ok

z_big = reparameterize(mu_big, logvar_big, big_eps)
z_wrong = mu_big + torch.exp(logvar_big) * big_eps   # variant: exp(logvar) used as the std
print(z_big.mean(dim=0), z_big.var(dim=0, correction=0), z_wrong.var(dim=0, correction=0))

try:
    reparameterize(torch.zeros(2, 3), torch.zeros(2, 3), torch.zeros(3, 3))
    mismatch_raises = False
except ValueError:
    mismatch_raises = True

### Answer check

In [ ]:
assert z.shape == (2, 3) and z.dtype == torch.float32
assert torch.allclose(z.detach(), expected_z, atol=ATOL, rtol=RTOL)
assert torch.allclose(mu.grad, torch.ones_like(mu), atol=ATOL, rtol=RTOL)
assert torch.allclose(logvar.grad, expected_logvar_grad, atol=ATOL, rtol=RTOL)
assert eps.grad is None
assert torch.allclose(mu2.grad, w, atol=ATOL, rtol=RTOL)
assert torch.allclose(logvar2.grad, w * 0.5 * torch.exp(0.5 * logvar2.detach()) * eps, atol=ATOL, rtol=RTOL)
assert distribution_ok(z_big)
assert not distribution_ok(z_wrong)
assert mismatch_raises